# Applying a QURI Parts circuit transpiler to a `Sub`

The `quri_parts.qsub.trans` package ships *qsub-native* transpilers — such as [`RZ2HSTTranspiler`](./transpiler.ipynb) — for the common fault-tolerant gate sets, each rewriting a `Sub` directly into a `Sub`. The broader [`quri_parts.circuit.transpile`](../../0_basics/8_transpiler/transpiler.ipynb) package offers many more transpilers that operate on a `QuantumCircuit` and have no qsub-native counterpart. `SeparateQURIPartsTranspiler` (from `quri_parts.qsub.trans.qp_trans`) is the bridge: it lets you run any `quri_parts.circuit` transpiler on a `Sub`.

Given a `Sub`, it walks the top-level operations and gathers maximal runs of *primitive* gates — those with a direct QURI Parts counterpart (`X`, `Y`, `Z`, `H`, `S`, `Sdag`, `SqrtX`/`SqrtXdag`, `SqrtY`/`SqrtYdag`, `T`, `Tdag`, `CNOT`, `CZ`, `SWAP`, `Toffoli`, `RX`, `RY`, `RZ`). Each run is converted to a `QuantumCircuit`, handed to the wrapped circuit transpiler(s), and spliced back into the `Sub`; any operation without a primitive counterpart, such as an un-expanded subroutine, is passed through untouched. This is exactly the mechanism the qsub-native transpilers use internally — `RZ2HSTTranspiler`, for example, is a thin wrapper around `SeparateQURIPartsTranspiler([RZ2HSTTranspiler()])`.

It assumes the qsub basics (`Op`, `Sub`, `SubBuilder`, `compile_sub`, and `QURIPartsEvaluatorHooks`); see the [qsub basics tutorial](./basics.ipynb) for a refresher.

As in the [gate-set tutorial](./transpiler.ipynb), we reuse one small helper. It compiles a transpiled `Sub` against a primitive gate set and returns a histogram of the gates in the resulting circuit. Compiling against a primitive set only succeeds if every operation resolves into those primitives, so the helper doubles as a check that the transpilation really landed in the target set.

In [ ]:
from collections import Counter

from quri_parts.qsub.compile import compile_sub
from quri_parts.qsub.machineinst import is_subcall


def gate_histogram(sub, primitives):
    compiled = compile_sub(sub, primitives)
    counter: Counter = Counter()

    def count(machine_sub):
        for op, _qubits, _registers in machine_sub.instructions:
            if is_subcall(op):
                count(op.sub)
            else:
                counter[op.op.id.local_name] += 1

    count(compiled)
    return counter

## Decomposing `CZ` into `H` and `CNOT`

`CZ2CNOTHTranspiler` (from `quri_parts.circuit.transpile`) rewrites every `CZ` as an `H`–`CNOT`–`H` sequence. It has no qsub-native wrapper, so we apply it to a `Sub` by passing it to `SeparateQURIPartsTranspiler`. Build a small `Sub`, transpile it, and confirm that no `CZ` survives — each `CZ` becomes two `H`s and a `CNOT`.

In [ ]:
from quri_parts.qsub.sub import SubBuilder
from quri_parts.qsub.lib.std import H, CZ
from quri_parts.qsub.lib import std
from quri_parts.qsub.trans.qp_trans import SeparateQURIPartsTranspiler
from quri_parts.circuit.transpile import CZ2CNOTHTranspiler

b = SubBuilder(3)
q0, q1, q2 = b.qubits
b.add_op(H, (q0,))
b.add_op(CZ, (q0, q1))
b.add_op(CZ, (q1, q2))
sub = b.build()
print("before:", gate_histogram(sub, (std.H, std.CZ)))

transpiler = SeparateQURIPartsTranspiler([CZ2CNOTHTranspiler()])
transpiled = transpiler(sub)
print("after: ", gate_histogram(transpiled, (std.H, std.CNOT)))

## Chaining several transpilers

`SeparateQURIPartsTranspiler` accepts a *sequence* of circuit transpilers and applies them in order to each run of primitive gates, so several rewrites compose in a single pass over the `Sub`. Here we first decompose `CZ` into `H` and `CNOT`, then rewrite every `H` into the `RZ`/`SqrtX` gate set with `H2RZSqrtXTranspiler`.

In [ ]:
from quri_parts.circuit.transpile import H2RZSqrtXTranspiler

b = SubBuilder(2)
q0, q1 = b.qubits
b.add_op(H, (q0,))
b.add_op(CZ, (q0, q1))
sub = b.build()
print("before:", gate_histogram(sub, (std.H, std.CZ)))

transpiler = SeparateQURIPartsTranspiler([CZ2CNOTHTranspiler(), H2RZSqrtXTranspiler()])
transpiled = transpiler(sub)
print("after: ", gate_histogram(transpiled, (std.RZ, std.SqrtX, std.CNOT)))